# Inverse Design: Learning the Wind
This notebook provides a detailed, mathematically rigorous walkthrough of the **Holonomic Association Model (HAM)**.
The HAM library treats differentiable **Finsler Geometries** as learnable modules in JAX, allowing us to solve optimal control and simulation problems purely through geometric dynamics.

### Core Philosophy: Metric-First Design
In HAM, the `FinslerMetric` object is the single source of truth. We define the scalar **Energy Functional**:
$$\mathcal{E}[\gamma] = \int \frac{1}{2} F^2(x, \dot{x}) dt$$
where $F(x,v)$ is the Finsler cost function. Everything else—including the Geodesic Spray (the physics engine), curvature, and Berwald parallel transport—is auto-differentiated directly from $F$ using `jax.grad` and `jax.hessian`, entirely avoiding the $O(N^3)$ computational cost of expanding Christoffel symbols manually.

## 1. Setup and Inverse Problem Definition
One of the Holonomic Association Model's most powerful features is its differentiable architecture. Because the metrics and solvers (like AVBD) are fully written in JAX, we can compute gradients *through* the optimal path-finding process.

This enables us to solve **inverse geometric problems**: if we observe agents taking specific optimal paths, can we deduce the underlying vector field (wind) they are experiencing?


In [1]:
import jax.numpy as jnp
import jax
import optax
import plotly.graph_objects as go
import numpy as np
from jax import config

from ham.geometry import Sphere, Randers
from ham.solvers import AVBDSolver
from ham.vis import generate_icosphere


## 2. Generating Ground Truth Observations
We establish a ground truth wind field (a vortex) and generate 4 trajectory observations. We assume we know the start and end points of these trajectories, but we *don't* know the wind field generating the curvature.


In [2]:
def vortex_field(x): 
    center = jnp.array([0.0, 1.0, 0.0])
    return 0.8 * jnp.cross(center, x)

sphere = Sphere(radius=1.0)
true_metric = Randers(sphere, lambda x: jnp.eye(3), vortex_field)
solver = AVBDSolver(step_size=0.05, beta=5.0, iterations=100)

# True Trajectories (Observations)
starts = [jnp.array([1.0, 0.0, 0.0]), jnp.array([0.0, 0.0, 1.0]), 
          jnp.array([-1.0, 0.0, 0.0]), jnp.array([0.0, 0.0, -1.0])]
ends   = [jnp.array([0.0, 0.0, -1.0]), jnp.array([1.0, 0.0, 0.0]), 
          jnp.array([0.0, 0.0, 1.0]), jnp.array([-1.0, 0.0, 0.0])]

true_trajs = [solver.solve(true_metric, s, e, n_steps=20) for s, e in zip(starts, ends)]


## 3. Parameterizing the Unknown Vector Field
We parameterize our learned wind field with a simple 3D vector parameter `w_param`. 
Our Finsler metric function is dynamically reconstructed on every forward pass using this trainable parameter.


In [3]:
w_param = jnp.array([0.0, 0.0, 0.0]) # Initial guess: No wind

def make_metric(w):
    # Reconstruct the wind field from the learned vector
    def learned_wind(x): return jnp.cross(w, x)
    return Randers(sphere, lambda x: jnp.eye(3), learned_wind)


## 4. Differentiable Loss Function
We define an L2 loss between the vertices of our currently predicted paths and the ground truth paths. 
Because `AVBDSolver.solve` utilizes continuous gradient descents that are differentiable themselves, we can invoke `jax.value_and_grad` on this loss function directly!


In [4]:
@jax.jit
def loss_fn(w):
    metric = make_metric(w)
    loss = 0.0
    for i, (s, e) in enumerate(zip(starts, ends)):
        pred_traj = solver.solve(metric, s, e, n_steps=20)
        loss += jnp.mean((pred_traj.xs - true_trajs[i].xs)**2)
    return loss

loss_and_grad = jax.value_and_grad(loss_fn)


## 5. Gradient Descent (Learning the Wind)
We use the Adam optimizer from `optax` to iteratively backpropagate through the AVBD solver, adjusting the underlying Zermelo wind parameter to minimize the path discrepancies.


In [7]:
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(w_param)

print("Learning the wind field...")
for step in range(100):
    loss, grads = loss_and_grad(w_param)
    updates, opt_state = optimizer.update(grads, opt_state)
    w_param = optax.apply_updates(w_param, updates)
    if step % 5 == 0:
        print(f"Step {step:02d} | Loss: {loss:.4f} | Learned Wind Param: {w_param}")

print(f"\nTrue Wind Param: [0.0, 1.0, 0.0] * 0.8 = [0.0, 0.8, 0.0]")
print(f"Learned Param:   {w_param}")


Learning the wind field...
Step 00 | Loss: 0.0000 | Learned Wind Param: [0.08297671 1.0110996  0.07060914]
Step 05 | Loss: 0.0000 | Learned Wind Param: [0.02536495 0.72401404 0.02838409]
Step 10 | Loss: 0.0000 | Learned Wind Param: [-0.02996759  0.8760287  -0.02891546]
Step 15 | Loss: 0.0000 | Learned Wind Param: [0.02043387 0.81451786 0.0206019 ]
Step 20 | Loss: 0.0000 | Learned Wind Param: [-0.01832559  0.7648276  -0.00940128]
Step 25 | Loss: 0.0000 | Learned Wind Param: [0.01561945 0.83357495 0.00131653]
Step 30 | Loss: 0.0000 | Learned Wind Param: [-0.01189936  0.7980137   0.00380529]
Step 35 | Loss: 0.0000 | Learned Wind Param: [ 0.00782751  0.78755003 -0.00729823]
Step 40 | Loss: 0.0000 | Learned Wind Param: [-0.00396844  0.8169706   0.00550206]
Step 45 | Loss: 0.0000 | Learned Wind Param: [ 6.7199580e-05  7.9028213e-01 -8.8689267e-04]
Step 50 | Loss: 0.0000 | Learned Wind Param: [ 0.0029044   0.8027305  -0.00293959]
Step 55 | Loss: 0.0000 | Learned Wind Param: [-0.00342325  0.80

## 6. Interactive 3D Visualization
We use Plotly to render the true data vs the learned paths, alongside the inferred wind field. Rotate the interactive render to verify that our `w_param` vector successfully induces the expected rotational drift!


In [13]:
learned_metric = make_metric(w_param)
learned_trajs = [solver.solve(learned_metric, s, e, n_steps=20) for s, e in zip(starts, ends)]

fig = go.Figure()
pts, faces = generate_icosphere(radius=1.0, subdivisions=2)

# Plot sphere
fig.add_trace(go.Mesh3d(
    x=pts[:,0], y=pts[:,1], z=pts[:,2],
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    color='white', opacity=0.2, alphahull=0
))

# Plot Learned Wind
def get_wind(pt):
    H, W, lam = learned_metric.zermelo_data(pt)
    return W
learned_vecs = np.array(jax.vmap(get_wind)(pts))
fig.add_trace(go.Cone(
    x=pts[:,0], y=pts[:,1], z=pts[:,2],
    u=learned_vecs[:,0], v=learned_vecs[:,1], w=learned_vecs[:,2],
    sizemode='absolute', sizeref=0.3, colorscale='Oranges', showscale=False, name='Learned Wind'
))

# Plot True vs Learned Paths
for i, (t_true, t_learn) in enumerate(zip(true_trajs, learned_trajs)):
    fig.add_trace(go.Scatter3d(
        x=t_true.xs[:,0], y=t_true.xs[:,1], z=t_true.xs[:,2],
        mode='lines', line=dict(color='black', width=6, dash='dash'), 
        name='True Data',
        showlegend=(i == 0)
    ))
    fig.add_trace(go.Scatter3d(
        x=t_learn.xs[:,0], y=t_learn.xs[:,1], z=t_learn.xs[:,2],
        mode='lines', line=dict(color='red', width=4), 
        name='Learned',
        showlegend =(i==0)
    ))

fig.update_layout(title="Inverse Design: Recovering Zermelo Wind from Trajectories", scene=dict(aspectmode='data'))
fig.show()
